# RAG以及mlivus预处理

In [1]:
#操作客户端
from pymilvus import MilvusClient
#默认端口号为19530
client=MilvusClient("http://localhost:19530")

### 列出所有数据库

In [102]:
existed_databases = client.list_databases()
for x in existed_databases:
    print(x)

default
rag_demo


### 创建数据库

In [ ]:
# 若不存在则创建
db_name = "weather"
existed_databases = client.list_databases()
if db_name not in existed_databases:
    client.create_database(db_name=db_name)
    print(f"数据库 '{db_name}' 已创建")
else:
    print(f"数据库 '{db_name}' 已存在")

### 删除对应数据库

In [82]:
client.drop_database("")

### 切换当下操作的数据库

In [77]:
#当下操作的是rag_damo数据库
client.use_database(db_name="")

### 查看数据库下的collection

In [ ]:
collection=client.list_collections()
for x in collection:
    print(x)

### 删除数据库下的collection

In [81]:
client.drop_collection(collection_name="radar_kb")

In [56]:
#环境变量的配置
from dotenv import load_dotenv
load_dotenv()

True

In [57]:
#百炼原生接口，由阿里官方 SDK 封装自动，从环境变量 DASHSCOPE_API_KEY 读取密钥，不用手动传依赖包：langchain-community（内部封装了 dashscope SDK）

from langchain_community.embeddings import DashScopeEmbeddings

# 创建 DashScope Embeddings 实例
embed_model = DashScopeEmbeddings(
    model="text-embedding-v4",  # 阿里云百炼 embedding 模型
)

# 详细实现

In [58]:
# LLM 客户端
import os
from openai import OpenAI

llm_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)
LLM_MODEL = "deepseek-chat"
print("LLM客户端初始化完成，模型:", LLM_MODEL)

# 原始数据
def build_forecast_data_text():
    return """【雷达数据分析结果】
- 时次: 2026-06-16 12:00
- 数据来源: 雷达实况 1个文件, 雷达外推 1个文件
- 总时间跨度: 1小时 (当前雷达实况 + 未来1-2小时外推)

【一、雷达实况分析】
- 深圳内对流单体信息
  - 识别到对流单体: 0个
- 深圳实际反射率覆盖区域统计
  基于实际反射率（18dBZ以上）分析全市及各区（含深汕）覆盖率，输出信息包括：区域名称、覆盖率、强度。
  - 深圳（全市）:覆盖率:90.3%, 最大反射率:47dBZ, 平均反射率:32dBZ, 强度:强对流
  - 福田:覆盖率:64.8%, 最大反射率:43dBZ, 平均反射率:38dBZ, 强度:强
  - 光明:覆盖率:57.1%, 最大反射率:47dBZ, 平均反射率:38dBZ, 强度:强对流
  - 龙华:覆盖率:50.7%, 最大反射率:44dBZ, 平均反射率:29dBZ, 强度:强
  - 大鹏:覆盖率:50.3%, 最大反射率:34dBZ, 平均反射率:25dBZ, 强度:中等
  - 罗湖:覆盖率:49.3%, 最大反射率:42dBZ, 平均反射率:36dBZ, 强度:强
  - 坪山:覆盖率:48.6%, 最大反射率:37dBZ, 平均反射率:30dBZ, 强度:强
  - 盐田:覆盖率:40.6%, 最大反射率:39dBZ, 平均反射率:35dBZ, 强度:强
  - 宝安:覆盖率:37.3%, 最大反射率:46dBZ, 平均反射率:29dBZ, 强度:强对流
  - 龙岗:覆盖率:30.2%, 最大反射率:44dBZ, 平均反射率:29dBZ, 强度:强
  - 南山:覆盖率:27.7%, 最大反射率:42dBZ, 平均反射率:35dBZ, 强度:强
  - 深汕:覆盖率:96.8%, 最大反射率:46dBZ, 平均反射率:34dBZ, 强度:强对流
- 深圳周边对流单体信息
  - 识别到对流单体: 20个
  需要输出所有单体信息。
  - ID:150,区域:珠江口,强度:58.0dBZ
  - ID:158,区域:珠江口,强度:57.0dBZ
  - ID:6,区域:珠江口,强度:55.0dBZ
  - ID:1,区域:珠江口,强度:52.0dBZ
  - ID:11,区域:香港,强度:51.0dBZ
  - ID:236,区域:珠江口,强度:51.0dBZ
  - ID:64,区域:珠江口,强度:50.0dBZ
  - ID:164,区域:珠江口,强度:50.0dBZ
  - ID:21,区域:珠江口,强度:49.0dBZ
  - ID:96,区域:珠江口,强度:48.0dBZ
  - ID:36,区域:珠江口,强度:47.0dBZ
  - ID:192,区域:珠江口,强度:47.0dBZ
  - ID:179,区域:珠江口,强度:45.0dBZ
  - ID:386,区域:珠江口,强度:45.0dBZ
  - ID:15,区域:珠江口,强度:44.0dBZ
  - ID:110,区域:珠江口,强度:44.0dBZ
  - ID:122,区域:珠江口,强度:44.0dBZ
  - ID:34,区域:珠江口,强度:42.0dBZ
  - ID:40,区域:珠江口,强度:42.0dBZ
  - ID:59,区域:珠江口,强度:42.0dBZ

【二、雷达外推与降雨分析】
- 雷达外推分析
  - 强度分析
   - 反射率序列: 58.0 -> 56.0 -> 56.0 -> 56.0 -> 56.0 -> 56.0 -> 56.0 -> 56.0 -> 55.0 -> 56.0 -> 56.0 -> 56.0 -> 56.0 -> 55.0 -> 54.0 -> 56.0 -> 54.0 -> 55.0 -> 56.0 -> 55.0 -> 56.0
   - 反射率变化: 0.0dBZ
  -移动分析
   - 移动方向: 东南
   - 移动速度: 20 km/h
   - 移动趋势判断: 向东南缓慢移动
  - 影响覆盖面分析
   - 连续覆盖率序列(%): 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0 -> 100.0
   - 覆盖面变化: 0.0%
  - 云团形态识别
   - 典型形态: 块状强回波
   - 形态描述: 强对流天气，可能有雷暴
【降雨预报分析】
  -【未来2小时降雨强度趋势分析】(12:06-14:00)
   - 全市平均雨量变化时序（mm）:2.5, 2.4, 2.4, 2.2, 2.3, 2.4, 2.5, 2.7, 2.7, 2.7, 2.7, 2.7, 2.7, 2.7, 2.6, 2.6, 2.6, 2.6, 2.5, 2.4
  -【未来1小时降雨预报】(13:00)
分析方法：输出深圳全市及各区（含深汕）降雨信息，包括：覆盖率、最大、最小、平均、降雨级别
   - 深圳（全市）:覆盖100.1%, 最大67.8毫米, 最小0.1毫米, 平均8.4毫米, 大暴雨
   - 光明: 覆盖100.0%, 最大67.8毫米, 最小2.9毫米, 平均44.3毫米, 暴雨
   - 宝安: 覆盖98.9%, 最大62.6毫米, 最小0.6毫米, 平均17.4毫米, 暴雨
   - 龙华: 覆盖97.4%, 最大58.7毫米, 最小4.9毫米, 平均30.7毫米, 暴雨
   - 龙岗: 覆盖100.0%, 最大30.7毫米, 最小0.1毫米, 平均4.5毫米, 暴雨
   - 南山: 覆盖100.0%, 最大20.8毫米, 最小0.1毫米, 平均5.6毫米, 大雨
   - 福田: 覆盖100.0%, 最大6.2毫米, 最小4.3毫米, 平均5.0毫米, 中雨
   - 罗湖: 覆盖100.0%, 最大5.8毫米, 最小5.0毫米, 平均5.4毫米, 中雨
   - 盐田: 覆盖100.0%, 最大5.6毫米, 最小4.7毫米, 平均5.0毫米, 中雨
   - 坪山: 覆盖100.0%, 最大4.8毫米, 最小2.5毫米, 平均4.0毫米, 小雨
   - 深汕: 覆盖100.0%, 最大4.1毫米, 最小0.2毫米, 平均1.8毫米, 小雨
  - 大鹏: 覆盖98.8%, 最大2.2毫米, 最小0.1毫米, 平均0.5毫米, 小雨

-【未来2小时降雨预报】(14:00)
分析方法：输出深圳全市及各区（含深汕）降雨信息，包括：覆盖率、最大、最小、平均、降雨级别
  - 深圳（全市）:覆盖100.1%, 最大44.9毫米, 最小0.4毫米, 平均6.2毫米, 暴雨
  - 光明: 覆盖100.0%, 最大44.9毫米, 最小4.2毫米, 平均30.3毫米, 暴雨
  - 龙华: 覆盖97.4%, 最大40.4毫米, 最小5.7毫米, 平均20.5毫米, 暴雨
  - 宝安: 覆盖98.9%, 最大36.1毫米, 最小2.2毫米, 平均10.2毫米, 暴雨
  - 龙岗: 覆盖100.0%, 最大13.0毫米, 最小0.9毫米, 平均4.5毫米, 中雨
  - 南山: 覆盖100.0%, 最大7.9毫米, 最小0.8毫米, 平均4.9毫米, 中雨
  - 福田: 覆盖100.0%, 最大6.8毫米, 最小5.0毫米, 平均6.0毫米, 中雨
  - 罗湖: 覆盖100.0%, 最大6.5毫米, 最小5.7毫米, 平均6.0毫米, 中雨
  - 盐田: 覆盖100.0%, 最大5.9毫米, 最小5.4毫米, 平均5.7毫米, 中雨
  - 坪山: 覆盖100.0%, 最大5.9毫米, 最小4.0毫米, 平均5.1毫米, 中雨
  - 大鹏: 覆盖98.8%, 最大4.4毫米, 最小0.9毫米, 平均2.4毫米, 小雨
  - 深汕: 覆盖100.0%, 最大2.7毫米, 最小0.4毫米, 平均1.0毫米, 小雨
"""
# 系统提示词
SYSTEM_PROMPT = """你是深圳市气象台的预报员，请根据下面的数据分析数据，写一段面向决策服务的短临服务提示。内容要求如下：
## 核心规则
1. 使用通俗易懂的语言，不失气象专业与严谨，结构清晰，重点突出，简单明了。只需要一段话。
2. 关于雷达回波的描述：以深圳为中心，只需要一句话概括。不要具体数字，以文字描述为主。不需要反射率数据，不需要移动速度数据等。
3. 当深圳已出现强回波时，简单研判是否会加强；反之，则监测深圳周边的雷达回波，简单研判多久会影响深圳。
4. 内容构成：
   - 第一句话：总体分析（根据全部数据一句话描述全市天气形势，不要用时效类话术）。
   - 第二句：降雨预报（以"预计未来x-x小时"开头，可按总体-重点（达到暴雨的）-次要的顺序分析总结；可按降雨级别、行政区域进行归类分析；逻辑严谨，降雨级别不能改，跨级别描述，小的放前面）。
5. 把未来天气情况放到一起，不要重复使用"未来"、"预计"等字样（除第二句必需的"预计未来x-x小时"开头外）。
6. 按区域描述的不带具体行政区名称（用"中西部""南部""北部""东部""西部"等大区名称代替）。
7. 雨量数据不能自行估测，降雨量数值规范：直接去掉小数部分，精确到个位，个位接近1的个位取1，个位接近5的个位取5，接近10的个位取0十位进1，区间值中间用"—"隔开。
8. 区域划分：中部(罗湖福田)、西部(宝安光明南山龙华)、东部(盐田坪山大鹏)、南部(南山福田罗湖盐田大鹏)、北部(光明龙华龙岗坪山)、中西部(罗湖福田宝安光明南山龙华)、深汕。
9. 降雨等级标准：中雨(5.0≤R≤14.9)、大雨(15.0≤R≤29.9)、暴雨(30.0≤R)、阵雨/小雨(0.1≤R≤4.9)。
10. 各区域不能重复出现。
11. 重复生成时，描述顺序与风格尽可能保持一致。

## 多轮对话
用户可能会对生成的预报文本提出修改意见，你需要根据意见修改文本，同时保持其他内容不变。常见请求：
- "简单点"/"精简一点" → 缩短文本，保留核心信息
- "把预报雨量调大点" → 在合理范围内将雨量值略上调
- "暴雨重点说" → 突出暴雨区域描述
- "换一种说法" → 保持内容但调整表达方式

请只返回最终的预报段落文本，不要加引号或额外解释。"""

LLM客户端初始化完成，模型: deepseek-chat


In [62]:
# ForecastChat — 对话管理，支持生成与多轮修改
class ForecastChat:

    def __init__(self, system_prompt=SYSTEM_PROMPT, model=LLM_MODEL):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        #初始化实例变量，表示当前还没有生成任何预报，后续表示当前预测结果
        self.current_forecast = None
        self.history = []

    def _call_llm(self):
        """调模型拿回复，带异常处理"""
        try:
            response = llm_client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                temperature=0.7,
                max_tokens=2048
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"[LLM调用失败] {e}"

    def generate(self, forecast_data_text):
        #短临数据生成一段话 ，后续可放入RAG
        user_msg = f"请根据以下数据，按规则写一段短临服务提示：\n\n{forecast_data_text}"
        self.messages.append({"role": "user", "content": user_msg})
        reply = self._call_llm()
        self.messages.append({"role": "assistant", "content": reply})
        self.current_forecast = reply
        self.history.append({"action": "生成", "input": forecast_data_text[:50] + "...", "output": reply})
        return reply

    def refine(self, user_input):
        #多轮修改
        self.messages.append({"role": "user", "content": user_input})
        reply = self._call_llm()
        self.messages.append({"role": "assistant", "content": reply})
        self.current_forecast = reply
        self.history.append({"action": f"修改: {user_input}", "input": user_input, "output": reply})
        return reply

    def reset(self, system_prompt=None):
        #清空对话历史
        if system_prompt:
            self.messages = [{"role": "system", "content": system_prompt}]
        else:
            sys_msg = self.messages[0] if self.messages else {"role": "system", "content": SYSTEM_PROMPT}
            self.messages = [sys_msg]
        self.current_forecast = None
        self.history = []

    def get_current(self):
        #获取最新预报文本
        return self.current_forecast


print("ForecastChat 初始化完成")

ForecastChat 初始化完成


In [93]:
# 初始化对话管理器
chat = ForecastChat()

# 获取短临数据
data_text = build_forecast_data_text()

# 生成预报
print("短临预报:")
forecast = chat.generate(data_text)
print(forecast)

短临预报:
目前深圳受强对流云团影响，全市大部已出现强降雨，珠江口一带仍有多个强对流单体向东南方向移动并持续影响我市。预计未来1-2小时，降雨总体维持强盛，其中西部和中西部有暴雨，局部可达大暴雨量级；南部和中部以中到大雨为主；东部和深汕以小雨到中雨为主。


In [94]:
#多轮修改

print("修改1：简单点")
r1 = chat.refine("简单点，精简一点")
print(r1)

print("修改2：把预报雨量调大点")
r2 = chat.refine("把预报雨量调大点")
print(r2)

print("修改3：换一种说法")
r3 = chat.refine("换一种说法，用通俗易懂的语言")
print(r3)

修改1：简单点
目前深圳受强对流云团影响，全市降雨明显，珠江口强对流单体持续向东南移动。预计未来1-2小时，西部和中西部有暴雨，其他地区以中雨为主。


In [101]:
#输入
USER_INPUT = "未来一小时内，哪些城市会有大雨或者暴雨"

print(f"回答：{USER_INPUT}")
result = chat.refine(USER_INPUT)
print(result)

回答：未来一小时内，哪些城市会有大雨或者暴雨
未来1小时，西部和中西部有暴雨，南部有中雨，东部和深汕为小雨。


In [92]:
# 查看对话历史与当前最终结果

print(f"对话历史共 {len(chat.history)} 轮:\n")
for i, h in enumerate(chat.history):
    print(f"  第{i+1}轮: {h['action']}")

print("当前最新预报：")
print(chat.get_current())


对话历史共 0 轮:

当前最新预报：
None


In [91]:
#重置对话
chat.reset()
print("对话已重置")

对话已重置
